# Purpose


This notebook conducts statistical analysis and generates processed data underlying figure
that fit to empirical connectomes..

In [8]:
import pickle
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

import sys
sys.path.append('..')
from scripts import stats_analysis, indices, directed_modularity

In [2]:
tau = 0.05 # tau_reweight
prand = 0.1 # probability of random rewiring
K = 20 # number of instances
binary_flag = True  # define hubs by degree

In [3]:
# Import adjacency matrices of empirical connectomes
f = open('./Output/Experimental Adj.pckl', 'rb')
A_real = pickle.load(f)
f.close()

In [4]:
# Import resulted networks at the end of simulations
A_sim = {}

pin = 0.78
A_sim['C.elegans'] = {}
for k in range(K):
    f = open('./Output/CE_Ain_pin_'+str(pin)+'_prand_'+str(prand)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
    A_sim['C.elegans'][k], trunc_num = pickle.load(f)
    f.close()

pin = 0.15
A_sim['Mouse retina'] = {}
for k in range(K):
    f = open('./Output/MR_Cout_pin_'+str(pin)+'_prand_'+str(prand)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
    A_sim['Mouse retina'][k], trunc_num = pickle.load(f)
    f.close()

In [ ]:
# Monte Carlo test of goodness-of-fit of simulated networks
pvalues = []
for key in A_real.keys():
    pvalues.append(stats_analysis.Waller_gof(A_real[key], A_sim[key]))
pvalues

f = open('./Output/GOF_pvalues.pckl', 'wb')
pickle.dump(pvalues, f)
f.close()

In [ ]:
# Compute number of convergent and divergent hubs for empirical and simulated networks
thresh_ls = {} 
key = 'C.elegans'
thresh_ls[key] = [30, 40, 50, 60, 70, 80, 90, 100, 120] # threshold of hubs
key = 'Mouse retina'
thresh_ls[key] = [100, 200, 300, 400, 500] # threshold of hubs

real_con_num = {}; real_div_num = {}; con_num = {}; div_num = {}
for key in A_real.keys():
    real_con_num[key] = np.zeros(len(thresh_ls[key]))
    real_div_num[key] = np.zeros(len(thresh_ls[key]))
    con_num[key] = {}
    div_num[key] = {}
    for j, thresh in enumerate(thresh_ls[key]):
        # empirical networks
        real_con_num[key][j] = indices.hub_number(A_real[key], thresh, binary_flag, axisUsed=1)
        real_div_num[key][j] = indices.hub_number(A_real[key], thresh, binary_flag, axisUsed=0)
        
        # simulated networks
        con_num[key][thresh] = np.zeros(K)
        div_num[key][thresh] = np.zeros(K)
        for k in A_sim[key].keys():
            con_num[key][thresh][k] = indices.hub_number(A_sim[key][k], thresh, binary_flag, axisUsed=1)
            div_num[key][thresh][k] = indices.hub_number(A_sim[key][k], thresh, binary_flag, axisUsed=0)
f = open('./Output/hubs.pckl', 'wb')
pickle.dump([con_num, real_con_num, div_num, real_div_num], f)
f.close()

In [ ]:
# Compute connectivity for simulated networks and empirical networks
real_conn = {}; conn = {}

for key in A_real.keys():
    real_conn[key] = indices.connectedness(A_real[key])
    conn[key] = {}
    for k in A_sim[key].keys():     
        conn[key][k] = indices.connectedness(A_sim[key][k])
f = open('./Output/connectedness.pckl', 'wb')
pickle.dump([conn, real_conn], f)
f.close()

In [ ]:
# Obtain the cd units and intermediacte nodes for simulated and empirical networks
real_cd_pairs = {}; real_interms = {}
cd_pairs = {}; interms = {}
for key in A_real.keys():
    thresh = 0.2*A_real[key].shape[0]
    real_cd_pairs[key] = indices.cd_pairs(A_real[key], real_conn[key], thresh)
    real_interms[key] = indices.find_interm_subG(A_real[key], real_cd_pairs[key])

    cd_pairs[key] = {}; interms[key] = {}
    for k in A_sim[key].keys():     
        cd_pairs[key][k] = indices.cd_pairs(A_sim[key][k], conn[key][k], thresh)
        interms[key][k] = indices.find_interm_subG(A_sim[key][k], cd_pairs[key][k])
f = open('./Output/cdPairs.pckl', 'wb')
pickle.dump([cd_pairs, real_cd_pairs, interms, real_interms], f)
f.close()

In [ ]:
# Compute number of cd units, average size, average density and average weight of intermediate subgraphs of simulated and empirical networks
# Normalized by the density and average weight of the whole networks
real_subG_density = {}; real_subG_aveW = {}
subG_density = {}; subG_aveW = {}

for s, key in enumerate(A_real.keys()):
    density_real = np.sum(A_real[key]>0)/(A_real[key].shape[0] * (A_real[key].shape[0] - 1))
    aveW_real = np.sum(A_real[key])/(A_real[key].shape[0] * (A_real[key].shape[0] - 1))
    _, real_subG_density[key], real_subG_aveW[key] = indices.interm_stats(A_real[key], real_interms[key])
    real_subG_density[key] = real_subG_density[key]/density_real
    real_subG_aveW[key] = real_subG_aveW[key]/aveW_real
    
    subG_density[key] = np.zeros(K); subG_aveW[key] = np.zeros(K)
    for k in range(K):
        _, subG_density[key][k], subG_aveW[key][k] = indices.interm_stats(A_sim[key][k], interms[key][k])
        subG_density[key][k] = subG_density[key][k]/density_real
f = open('./Output/interm_stats.pckl', 'wb')
pickle.dump([subG_density, real_subG_density, subG_aveW, real_subG_aveW], f)
f.close()

In [5]:
# Import dictionaries of networks sampled at different steps in simulations for stationary analysis
A_sim = {}

pin = 0.78
A_sim['C.elegans'] = {}
for k in range(K):
    f = open('./Output_long/CE_Ain_pin_'+str(pin)+'_prand_'+str(prand)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
    A_sim['C.elegans'][k], trunc_num = pickle.load(f)
    f.close()

pin = 0.15
A_sim['Mouse retina'] = {}
for k in range(K):
    f = open('./Output_long/MR_Cout_pin_'+str(pin)+'_prand_'+str(prand)+'_tau_'+str(tau)+'_'+str(k)+'.pckl', 'rb')
    A_sim['Mouse retina'][k], trunc_num = pickle.load(f)
    f.close()

In [6]:
# Compute summary statistics
std = {}; skew = {}; kurt = {}
for key in A_sim.keys():
    std[key] = {}; skew[key] = {}; kurt[key] = {}
    for i, j in enumerate(A_sim[key][0].keys()):
        std[key][j] = np.zeros(K)
        skew[key][j] = np.zeros(K)
        kurt[key][j] = np.zeros(K)
            
for key in A_sim.keys():
    for k in range(K):
        for i, j in enumerate(A_sim[key][0].keys()):
            std[key][j][k], skew[key][j][k], kurt[key][j][k] = stats_analysis.stats_summaries(A_sim[key][k][j])

f = open('./Output_long/summary_stats_evol.pckl', 'wb')
pickle.dump([std, skew, kurt], f)
f.close()

In [9]:
# Compute connectivity metrics
cc = {}; ae = {}; mod = {}

for key in A_sim.keys():
    cc[key] = {}
    ae[key] = {}
    mod[key] = {} 
    for i, j in enumerate(A_sim[key][0].keys()):
        cc[key][j] = np.zeros(K)
        ae[key][j] = np.zeros(K)
        mod[key][j] = np.zeros(K)

for key in A_sim.keys():
    for k in range(K):
        for i, j in enumerate(A_sim[key][k].keys()):
            A = A_sim[key][k][j].copy()
            cc[key][j][k] = indices.cluster_coefficient(A)
            ae[key][j][k] = indices.average_efficiency(A)
            mod[key][j][k], _ = directed_modularity.getModularityIndex(A)

f = open('./Output_long/Topo_metrics_evol.pckl', 'wb')
pickle.dump([cc, ae, mod], f)
f.close()  